In [1]:
from bs4 import BeautifulSoup
import edgar
import os
import pathlib
import tarfile
import io
import zstd

In [2]:
def process_bulk_filings(year=None, quarter=None):
    data_dir = pathlib.Path(os.getenv("EDGAR_LOCAL_DATA_DIR"))
    base_url = 'https://www.sec.gov/Archives/edgar/Feed/'
    page = edgar.httprequests.get_with_retry(base_url)
    soup = BeautifulSoup(page, features="lxml")
    table = soup.find("table")

    if year:
        years = [year]
    else:
        years = []
        for tr in table.find_all('tr'):
            row = tr.find_all('td')
            if len(row) > 0:
                year = row[0].find('a').attrs['href'].replace("/", "")
                if year != "":
                    if int(year) >= 1997:
                        years.append(int(year))

        years.sort()

    for year in years:
        year_page = edgar.httprequests.get_with_retry(f"{base_url}{year}/")
        year_soup = BeautifulSoup(year_page, features="lxml")
        year_table = year_soup.find("table")
        if quarter:
            quarters = [f"QTR{quarter}"]
        else:
            quarters = []
            for tr in year_table.find_all('tr'):
                row = tr.find_all('td')
                if len(row) > 0:
                    qtr = row[0].find('a').attrs['href'].replace("/", "")
                    if qtr in ("QTR1", "QTR2", "QTR3", "QTR4"):
                        quarters.append(qtr)
        for quarter in quarters:
            quarter_page = edgar.httprequests.get_with_retry(f"{base_url}{year}/{quarter}/")
            quarter_soup = BeautifulSoup(quarter_page, features="lxml")
            quarter_table = quarter_soup.find("table")
            for tr in quarter_table.find_all('tr'):
                row = tr.find_all('td')
                if len(row) > 0:
                    filename = row[0].find('a').attrs['href']
                    if filename.split(".")[-1] == 'gz':
                        file_url = f"{base_url}{year}/{quarter}/{filename}"
                        file_data = edgar.httprequests.get_with_retry(file_url)
                        with tarfile.open(fileobj=io.BytesIO(file_data.content)) as tar:
                            for member in tar.getmembers():
                                f = tar.extractfile(member)
                                file_path = data_dir / "data" / str(year) / quarter / f"{member.name}.zstd"
                                content=zstd.compress(f.read())
                                try:
                                    file = file_path.open('wb')
                                except FileNotFoundError:
                                    file_path.parent.mkdir(parents=True, exist_ok=True)
                                    file = file_path.open('wb')
                                    file.write(content)
                                    file.flush()
                                file.close()
                            tar.close()